In [2]:
import pandas as pd
import numpy as np

In [5]:
# unzipping the kaggle download into data/raw/
import zipfile
import os

with zipfile.ZipFile('../data/raw/credit-card-customers.zip', 'r') as z:
    z.extractall('data/raw/')
print(os.listdir('data/raw/'))

['BankChurners.csv']


In [16]:
# Checking and adjusting raw data
df = pd.read_csv('data/raw/BankChurners.csv')
df.drop(columns=df.columns[-2:], inplace=True)
df = df.rename(columns={'CLIENTNUM': 'customer_id', 'Attrition_Flag': 'churn_flag'})
df['churn_flag'] = (df['churn_flag'] == 'Attrited Customer').astype(int)
df.info()
df.describe()
df.describe(include=['str'])

<class 'pandas.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CLIENTNUM                 10127 non-null  int64  
 1   Attrition_Flag            10127 non-null  str    
 2   Customer_Age              10127 non-null  int64  
 3   Gender                    10127 non-null  str    
 4   Dependent_count           10127 non-null  int64  
 5   Education_Level           10127 non-null  str    
 6   Marital_Status            10127 non-null  str    
 7   Income_Category           10127 non-null  str    
 8   Card_Category             10127 non-null  str    
 9   Months_on_book            10127 non-null  int64  
 10  Total_Relationship_Count  10127 non-null  int64  
 11  Months_Inactive_12_mon    10127 non-null  int64  
 12  Contacts_Count_12_mon     10127 non-null  int64  
 13  Credit_Limit              10127 non-null  float64
 14  Total_Revolving_B

,Attrition_Flag,Gender,Education_Level,Marital_Status,Income_Category,Card_Category
count,10127,10127,10127,10127,10127,10127
unique,2,2,7,4,6,4
top,Existing Customer,F,Graduate,Married,Less than $40K,Blue
freq,8500,5358,3128,4687,3561,9436


In [31]:
# Generating monthly spending activity table for each customer
records = []

for _, row in df.iterrows():
    cid      = row['customer_id']
    churned  = row['churn_flag']

    # Each customer has different spending behaviour with following randomness

    # 1. Each churned customer starts disengaging at a different month (5–11)
    #    Non-churners get month 99 — decay never triggers
    churn_start_month = np.random.randint(5, 12) if churned else 99

    # 2. Each churned customer has their own spending decay severity
    #    Some ghost (0.1), some just slow down (0.75)
    final_decay = np.random.uniform(0.1, 0.75) if churned else 1.0

    # 3. Seasonal multiplier — all customers spend more in month 11-12 towards year end
    seasonal = {1:0.85, 2:0.80, 3:0.90, 4:0.95, 5:1.00,
                6:1.00, 7:0.95, 8:0.95, 9:1.00, 10:1.05, 11:1.20, 12:1.30}

    # 4. Each customer has a personal activity baseline (some just transact more)
    personal_multiplier = np.random.normal(1.0, 0.15)  # ±15% personal baseline
    personal_multiplier = max(0.5, personal_multiplier)  # floor at 50%

    # 5. Monthly transaction base (annual total spread to monthly)
    monthly_txn_base   = row['Total_Trans_Ct']  / 12 * personal_multiplier
    monthly_amt_base   = row['Total_Trans_Amt'] / 12 * personal_multiplier
    monthly_login_base = np.random.randint(5, 15)  # customers vary: 5–15 logins/month

    for month in range(1, 13):
        # spending decay is gradual
        if month < churn_start_month:
            decay = 1.0
        else:
            # Linear ramp-down from 1.0 to final_decay over remaining months
            months_into_decay = month - churn_start_month
            total_decay_months = 12 - churn_start_month + 1
            progress = min(1.0, months_into_decay / max(total_decay_months, 1))
            decay = 1.0 - (1.0 - final_decay) * progress

        # Seasonal adjustment
        season = seasonal[month]

        # Occasional one-off spikes (big purchase, travel month)
        spike = np.random.choice([1.0, 1.0, 1.0, 1.0, 1.5, 2.0],
                                  p=[0.70, 0.10, 0.08, 0.07, 0.03, 0.02])

        # Generating transactions count, amount and number of logins during the month

        txn_count = max(0, int(
            np.random.poisson(max(0.1, monthly_txn_base * decay * season * spike))
        ))

        txn_amount = max(0, round(
            np.random.normal(monthly_amt_base * decay * season * spike,
                             monthly_amt_base * 0.2),   # std = 20% of base
            2
        ))

        login_count = max(0, int(
            np.random.poisson(max(0.1, monthly_login_base * decay))
        ))

        # Customer support contacts: churners escalate as decay deepens
        # The angrier/more disengaged they are, the more they call — then stop
        if churned and month >= churn_start_month:
            months_into_decay = month - churn_start_month
            if months_into_decay <= 2:
                # Early churn phase: calling support trying to fix things
                contact_probs = [0.30, 0.30, 0.20, 0.15, 0.05]
                contact_vals  = [0, 1, 2, 3, 4]
            else:
                # Late churn phase: gave up, stopped calling entirely
                contact_probs = [0.75, 0.15, 0.07, 0.02, 0.01]
                contact_vals  = [0, 1, 2, 3, 4]
        else:
            contact_probs = [0.70, 0.18, 0.08, 0.03, 0.01]
            contact_vals  = [0, 1, 2, 3, 4]

        support_contacts = np.random.choice(contact_vals, p=contact_probs)

        utilization_ratio = min(1.0, max(0, round(
            row['Avg_Utilization_Ratio'] * decay * np.random.normal(1, 0.12),
            3
        )))

        records.append({
            'customer_id':      cid,
            'month':            month,
            'txn_count':        txn_count,
            'txn_amount':       txn_amount,
            'login_count':      login_count,
            'support_contacts': support_contacts,
            'utilization_ratio':utilization_ratio,
            # for debugging / validation
            'churn_start_month': churn_start_month if churned else None,
            'final_decay':       round(final_decay, 3) if churned else None,
        })

monthly = pd.DataFrame(records)

In [33]:
monthly.describe()

,customer_id,month,txn_count,txn_amount,login_count,support_contacts,utilization_ratio,churn_start_month,final_decay
count,1.215240e+05,121524.000000,121524.000000,121524.000000,121524.000000,121524.000000,121524.000000,19524.000000,19524.000000
mean,7.391776e+08,6.500000,5.494520,373.281044,9.348762,0.496108,0.272133,7.992010,0.423892
std,3.690211e+07,3.452067,3.474821,330.905059,4.249944,0.867615,0.276972,2.002032,0.187450
min,7.080821e+08,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,0.101000
25%,7.130362e+08,3.750000,3.000000,166.740000,6.000000,0.000000,0.021000,6.000000,0.264000
50%,7.179264e+08,6.500000,5.000000,286.255000,9.000000,0.000000,0.172000,8.000000,0.425000
75%,7.731464e+08,9.250000,7.000000,438.640000,12.000000,1.000000,0.489000,10.000000,0.585000
max,8.283431e+08,12.000000,39.000000,4898.690000,31.000000,4.000000,1.000000,11.000000,0.750000


In [35]:
# some validation of the generated monthly data
validation = monthly.copy()
validation = validation.merge(df[['customer_id', 'churn_flag']], on='customer_id')

validation['period'] = validation['month'].apply(
    lambda m: 'late (9-12)' if m >= 9 else 'early (1-8)'
)

print(validation.groupby(['churn_flag', 'period'])['txn_count'].mean().unstack())

period      early (1-8)  late (9-12)
churn_flag                          
0              5.465662     6.729735
1              3.547711     3.236478
